In [ ]:
import re
import time
import requests

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings          
from langchain_core.tools import create_retriever_tool           
from langchain_ollama import ChatOllama
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor  
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.memory import ConversationBufferMemory

d:\AI Agent Mid_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
def scrape_all_products(base_url="https://colourpop.com", max_pages=20):
    """Fetch all products page by page until the feed runs out."""
    all_products = []
    page = 1
 
    while page <= max_pages:
        url = f"{base_url}/products.json?limit=250&page={page}"
        response = requests.get(url)
        data = response.json()
 
        products = data.get("products", [])
        if not products:
            break
 
        all_products.extend(products)
        print(f"Page {page}: fetched {len(products)} products (total: {len(all_products)})")
 
        page += 1
        time.sleep(0.5)  # be polite to the server, avoid hammering it
 
    return all_products
 

In [12]:
def clean_html(raw_html):
    """Strip HTML tags from a product description."""
    clean_text = re.sub(r"<[^>]+>", " ", raw_html)
    clean_text = re.sub(r"\s+", " ", clean_text).strip()
    return clean_text
 
 
def product_to_document(product):
    """Convert a raw product dict into a LangChain Document."""
    title = product.get("title", "")
    product_type = product.get("product_type", "")
    vendor = product.get("vendor", "")
    tags = ", ".join(product.get("tags", []))
    description = clean_html(product.get("body_html", ""))
 
    variants = product.get("variants", [])
    prices = [v.get("price") for v in variants if v.get("price")]
    price_range = (
        f"{min(prices)} - {max(prices)} USD"
        if len(set(prices)) > 1
        else f"{prices[0]} USD" if prices else "N/A"
    )
 
    in_stock = any(v.get("available") for v in variants)
 
    content = f"""Product: {title}
Type: {product_type}
Brand: {vendor}
Price: {price_range}
Availability: {"In Stock" if in_stock else "Out of Stock"}
Tags: {tags}
Description: {description}"""
 
    return Document(
        page_content=content,
        metadata={
            "title": title,
            "product_type": product_type,
            "price_range": price_range,
            "in_stock": in_stock,
            "source": product.get("handle", ""),
        },
    )
 

In [ ]:
def build_vectorstore(documents, save_path="faiss_hair_skin_products"):
    """Turn documents into embeddings and store them in FAISS."""
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    vectorstore = FAISS.from_documents(documents, embeddings)
    vectorstore.save_local(save_path)
    print(f"Vector store built and saved to '{save_path}' ")
    return vectorstore
 

In [13]:
def build_product_search_tool(vectorstore, k=5):
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
 
    tool = create_retriever_tool(
        retriever,
        name="search_hair_skin_products",
        description=(
            "Use this tool to search the hair and skin care product catalog. "
            "Useful for questions about prices, ingredients, product types, "
            "availability, or comparing products."
        ),
    )
    return tool
 
 
# ======================

In [ ]:
def build_agent(tools, model_name="llama3.2"):
    """
    """
    llm = ChatOllama(model=model_name, temperature=0)
 
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                (
                    "You are a helpful assistant specialized in hair and skin "
                    "care products. Always use the search tool before answering "
                    "any question about a product, price, or availability. "
                    "If a product is not found in the search results, say it is "
                    "not available in the catalog instead of making up information."
                ),
            ),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad"),
        ]
    )
 
    agent = create_tool_calling_agent(llm, tools, prompt)
 
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True,
    )
 
    agent_executor = AgentExecutor(
        agent=agent,
        tools=tools,
        memory=memory,
        verbose=True,  # shows how the agent reasons and when it calls the tool
    )
 
    return agent_executor
 
 

In [17]:
if __name__ == "__main__":
    # 1. Scrape the products (run once, then reuse the saved result)
    raw_products = scrape_all_products()
 
    # 2. Convert to Documents
    documents = [product_to_document(p) for p in raw_products]
    print(f"\nCreated {len(documents)} documents")
 
    # 3. Build the vector store
    vectorstore = build_vectorstore(documents)
 
    # 4. Build the tool
    tool = build_product_search_tool(vectorstore)
 
    # 5. Build the agent
    agent_executor = build_agent([tool], model_name="llama3.2")
 
    # 6. Try a conversation
    response1 = agent_executor.invoke(
        {"input": "Are there any moisturizing skin products under $20?"}
    )
    print("\n--- First answer ---")
    print(response1["output"])
 
    print("\n--- Follow-up question (tests memory) ---")
    response2 = agent_executor.invoke({"input": "Which one is the cheapest?"})
    print(response2["output"])
 

Page 1: fetched 250 products (total: 250)
Page 2: fetched 250 products (total: 500)
Page 3: fetched 250 products (total: 750)
Page 4: fetched 250 products (total: 1000)
Page 5: fetched 6 products (total: 1006)

Created 1006 documents


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2347.78it/s]


Vector store built and saved to 'faiss_hair_skin_products' ✅


> Entering new AgentExecutor chain...

Invoking: `search_hair_skin_products` with `{'query': 'moisturizing skin products under $20'}`


Product: In a Haze
Type: Cloud Whip Blurring Liquid Lipstick
Brand: ColourPop
Price: 10.00 USD
Availability: In Stock
Tags: #.upsell-price:900, 010226, color:Brown, event:$6 best sellers April 2026, event:Cloud Whip Liquid Lip 2025, Lips, Lipstick, Online Exclusive
Description: Your cool nude, elevated. ☁️ In a Haze is a soft, cool-toned nude that diffuses beautifully at the edges — effortless every time. Marshmallow Root Extract, Cloudberry Oil, and Hyaluronic Acid keep lips pillow-soft and never dry, all day. ✓ 100% Vegan &amp; Cruelty-Free

Product: Medium 12 W
Type: Pretty Fresh Tinted Moisturizer
Brand: ColourPop
Price: 15.00 USD
Availability: In Stock
Tags: #.collection:2023labordaysitewidesale, #.collection:PromoBackup2, #.collection:promoproducts, #.collection:TemptaliaPromo, #.hidd

In [19]:
# ==================================================================
# Hybrid Retrieval: BM25 (keyword) + FAISS (semantic) combined with RRF
# ==================================================================
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever

# 1. Sparse retriever (BM25) — matches exact words like "Moisturizer"
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 5

# 2. Dense retriever (FAISS) — matches by meaning (already built)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# 3. Combine both with Reciprocal Rank Fusion
#    weights=[0.5, 0.5] means both contribute equally — tune this later if needed
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5],
)

In [ ]:
from langchain_core.tools import create_retriever_tool

product_search_tool = create_retriever_tool(
    hybrid_retriever,  
    name="search_hair_skin_products",
    description=(
        "Use this tool to search the hair and skin care product catalog. "
        "Useful for questions about prices, ingredients, product types, "
        "availability, or comparing products."
    ),
)

tools = [product_search_tool]

In [21]:
bm25_retriever.invoke("moisturizer")

[Document(metadata={'title': 'Fair 1 N', 'product_type': 'Pretty Fresh Tinted Moisturizer', 'price_range': '15.00 USD', 'in_stock': True, 'source': 'fair-1-n'}, page_content='Product: Fair 1 N\nType: Pretty Fresh Tinted Moisturizer\nBrand: ColourPop\nPrice: 15.00 USD\nAvailability: In Stock\nTags: #.collection:2023labordaysitewidesale, #.collection:PromoBackup2, #.collection:promoproducts, #.collection:TemptaliaPromo, #.hidden:false, #.promo.30offsitewide, #.promo:complexion10, #.promo:GloskuPF50, #.promo:nicol10, #.promo:PFBOGO, #.promo:PFcomplexion, #.promo:prettyfreshupsell, #.promo:SinglesDay30, #.promo:treat, #.trigger:nationalnudedayflashsale, #.upsell-price:1000, #rmb.glosku:145, 092619, Alipay:May40, category:Face, color:Fair, event:Pretty Fresh, Fair, finish:Sheer, glosku:PFcomplexionBOGO, mvp.pretty-fresh-hyaluronic-acid-tinted-moisturizer, Natural, Neutral, Online Exclusive, productline:ColourPop, productsize:full, producttype:Pretty Fresh Tinted Moisturizer, Sheer, SKULifec

In [22]:
print("=== BM25 only ===")
for doc in bm25_retriever.invoke("cheapest moisturizing skin product"):
    print(doc.metadata["title"], "-", doc.metadata["product_type"])

print("\n=== Dense (FAISS) only ===")
for doc in dense_retriever.invoke("cheapest moisturizing skin product"):
    print(doc.metadata["title"], "-", doc.metadata["product_type"])

print("\n=== Hybrid (combined) ===")
for doc in hybrid_retriever.invoke("cheapest moisturizing skin product"):
    print(doc.metadata["title"], "-", doc.metadata["product_type"])

=== BM25 only ===
Kabuki Face and Body Brush - Makeup Brush
Body Kabuki - SOL Body Tools
Silicone Jelly Wand - Tools and Accessories
Silicone Blending Sponge - Makeup Tools & Accessories
Merriest Pout - Lip Set

=== Dense (FAISS) only ===
Medium 12 W - Pretty Fresh Tinted Moisturizer
Medium Dark 15 W - Pretty Fresh Tinted Moisturizer
Medium Dark 14 W - Pretty Fresh Tinted Moisturizer
Dark 18 W - Pretty Fresh Tinted Moisturizer
Medium Dark 13 W - Pretty Fresh Tinted Moisturizer

=== Hybrid (combined) ===
Kabuki Face and Body Brush - Makeup Brush
Medium 12 W - Pretty Fresh Tinted Moisturizer
Body Kabuki - SOL Body Tools
Medium Dark 15 W - Pretty Fresh Tinted Moisturizer
Silicone Jelly Wand - Tools and Accessories
Medium Dark 14 W - Pretty Fresh Tinted Moisturizer
Silicone Blending Sponge - Makeup Tools & Accessories
Dark 18 W - Pretty Fresh Tinted Moisturizer
Merriest Pout - Lip Set
Medium Dark 13 W - Pretty Fresh Tinted Moisturizer
